## Chat Completions API

Send a conversation, get the next message. The `openai` package is just a wrapper around an HTTP endpoint.

In [ ]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import requests
import os

load_dotenv()

## Call the endpoint with requests

In [ ]:
endpoint = os.getenv("GPT_ENDPOINT").rstrip("/")
model = os.getenv("GPT_MODEL", "gpt-4o")
url = f"{endpoint}/openai/deployments/{model}/chat/completions?api-version={os.getenv('GPT_API_VERSION')}"

response = requests.post(
    url,
    headers={"api-key": os.getenv("GPT_KEY"), "Content-Type": "application/json"},
    json={"messages": [{"role": "user", "content": "Tell me a fun fact."}]},
    timeout=120,
)

response.json()

In [ ]:
print(response.json()["choices"][0]["message"]["content"])

## Same call with the client library

In [ ]:
client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)

resp = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Tell me a fun fact."}],
    timeout=120.0,
)

print(resp.choices[0].message.content)

## Multi turn conversation

The API is stateless, so past turns go back in the `messages` list every time.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Name one planet."},
]

resp = client.chat.completions.create(model=model, messages=messages, timeout=120.0)
messages.append({"role": "assistant", "content": resp.choices[0].message.content})
messages.append({"role": "user", "content": "How many moons does it have?"})

resp = client.chat.completions.create(model=model, messages=messages, timeout=120.0)
print(resp.choices[0].message.content)

## Streaming

In [ ]:
stream = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Explain tokens in two sentences."}],
    stream=True,
    timeout=120.0,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")